In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
attributions_path = "/work/results/model_attributions/"
model_name = "chronos-t5-base"
run_config = "sint_1_sind_0_clen_512_plen_512_nsamp_1_tidx_0_4096_tidx_0_512_seed_42"

data_path = os.path.join(attributions_path, model_name, run_config)

In [ ]:
layers = [i for i in range(12)]
residual_logit_maps = {}
mlp_logit_maps = {}
sa_logit_maps = {}
ca_logit_maps = {}
for layer in layers:
    # load in numpy array from data_path/layer_{layer}/data/read-stream-full_{layer}.npy
    residual_logit_maps[layer] = np.load(os.path.join(data_path, f"layer_{layer}", "data", f"mlp-input_{layer}.npy"))
    mlp_logit_maps[layer] = np.load(os.path.join(data_path, f"layer_{layer}", "data", f"mlp-output_{layer}.npy"))
    sa_logit_maps[layer] = np.load(os.path.join(data_path, f"layer_{layer}", "data", f"sa-input_{layer}.npy"))
    ca_logit_maps[layer] = np.load(os.path.join(data_path, f"layer_{layer}", "data", f"ca-input_{layer}.npy"))

In [ ]:
layer = 11

resid_map = residual_logit_maps[layer].T
sa_map = sa_logit_maps[layer].T
ca_map = ca_logit_maps[layer].T
mlp_map = mlp_logit_maps[layer].T
# Calculate residual minus MLP contribution
diff_map = mlp_map - resid_map

fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# Determine common color scale for all plots
vmin = min(resid_map.min(), mlp_map.min(), diff_map.min())
vmax = max(resid_map.max(), mlp_map.max(), diff_map.max())

# Upper subplot - residual stream
im0 = axs[0].imshow(resid_map, aspect="auto", origin="lower", vmin=vmin, vmax=vmax)
axs[0].set_title("Residual Stream Logit Map")

# Middle subplot - MLP contribution
im1 = axs[1].imshow(mlp_map, aspect="auto", origin="lower", vmin=vmin, vmax=vmax)
axs[1].set_title("MLP Contribution Logit Map")

# Lower subplot - Residual minus MLP
im2 = axs[2].imshow(diff_map, aspect="auto", origin="lower", vmin=vmin, vmax=vmax)
axs[2].set_title("Residual Stream minus MLP Contribution")

# Add a single colorbar for all plots
cbar = fig.colorbar(im0, ax=axs, location="right", shrink=0.8)

plt.show()